### Transformação de dados na Camada Gold

Leitura dos dados da camada Silver, aplicação das regras de negócio KPI

In [0]:
# CARREGAMENTO DE VARIÁVEIS DE AMBIENTE
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, current_timestamp, year, month, trim, when, lit, row_number, to_date, to_timestamp,coalesce
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, BooleanType, TimestampType
from delta.tables import DeltaTable

load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")

# DICIONÁRIO DE CREDENCIAIS OAUTH PARA USAR COM SPARK
adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# DICIONÁRIO DE PATHS
paths = {
    "food_avaliacoes_produto_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_avaliacoes_produto",
    "food_estoque_lojas_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_estoque_lojas",
    "food_avaliacoes_produto_gold": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_avaliacoes_produto",
    "food_estoque_lojas_gold": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_estoque_lojas",
    "raw_path": f"abfss://raw@{storage_account_name}.dfs.core.windows.net/batch-data"
}


In [0]:
# LEITURA DAS TABELAS BRONZE
df_silver_avaliacoes = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_avaliacoes_produto_silver"])
)
 
df_silver_estoque = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_estoque_lojas_silver"])
)

df_silver_avaliacoes.printSchema()
df_silver_estoque.printSchema()
display(df_silver_avaliacoes.limit(5))
display(df_silver_estoque.limit(5))